# Milestone 3: Clinical Evaluation, Calibration Reliability & Quantitative Grad-CAM

This runnable walkthrough demonstrates the Milestone 3 evaluation and explainability suite in **MRITumorDetection-CNN**:
1. **Stage 1 Binary Screening**: Default `0.50` threshold vs. sensitivity-calibrated threshold (`find_sensitivity_threshold(..., target_sensitivity=0.98)`), confusion matrix, and ROC curve.
2. **Stage 2 Subtype Calibration & Abstention**: 3-class evaluation (`Macro-F1`, `Balanced Accuracy`, `ECE`, `Brier Score`, `selective_accuracy`, `abstention_rate`) before vs. after `TemperatureScaler`.
3. **End-to-End `4x5` Clinical Routing Matrix**: Unified cascade evaluation (`compute_cascade_metrics`) comparing `overall_4way_*`, `selective_4way_*`, `cascade_tumor_sensitivity`, and `cascade_healthy_specificity`.
4. **Grad-CAM & Quantitative Mask Validation**: Comparing `SimpleCNN`, `TimmClassifier("efficientnet_b0")`, and `TimmClassifier("resnet50")` (`pretrained=False`) on synthetic `(224, 224)` MRI slices with circular ground-truth tumor masks, computing `compute_pointing_game_hit`, `compute_energy_inside_mask`, `evaluate_gradcam_against_masks`, `overlay_gradcam`, `plot_gradcam_examples`, `plot_ablation_comparison`, and `generate_evaluation_report`.

## 1. Imports, Reproducibility & Temporary Scratch Directory

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

# Ensure src/ is importable if the notebook is run without editable install
repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from mri_tumor.evaluation import (
    compute_cascade_metrics,
    compute_ece,
    compute_stage1_metrics,
    compute_stage2_metrics,
    generate_evaluation_report,
    plot_ablation_comparison,
    plot_calibration_curve,
    plot_confusion_matrix,
    plot_gradcam_examples,
    plot_roc_curves,
)
from mri_tumor.explain import (
    GradCAM,
    compute_energy_inside_mask,
    compute_pointing_game_hit,
    evaluate_gradcam_against_masks,
    overlay_gradcam,
)
from mri_tumor.models import SimpleCNN, TimmClassifier
from mri_tumor.training import (
    ABSTAIN_LABEL,
    TemperatureScaler,
    apply_temperature_and_abstention,
    find_sensitivity_threshold,
    set_seed,
)

set_seed(42)
scratch_ctx = tempfile.TemporaryDirectory(prefix="mri_milestone3_")
scratch_dir = Path(scratch_ctx.name)
plot_paths: dict[str, Path] = {}

def show_saved_plot(path: Path, figsize: tuple[float, float] = (6.5, 5.0)) -> None:
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

print(f"PyTorch: {torch.__version__} | Scratch report dir: {scratch_dir}")

## 2. Stage 1 Binary Screening: Default `0.50` vs. Sensitivity-Calibrated Threshold (`>= 0.98`)

We compare Stage 1 (`no_tumor` vs. `tumor`) at the default `0.50` probability cutoff against `find_sensitivity_threshold(..., target_sensitivity=0.98)` using `compute_stage1_metrics`, `plot_confusion_matrix`, and `plot_roc_curves`.

In [ ]:
rng = np.random.default_rng(42)
n_neg, n_pos = 240, 120
s1_targets = np.array([0] * n_neg + [1] * n_pos, dtype=np.int64)
neg_probs = np.clip(rng.beta(1.8, 7.0, size=n_neg), 0.005, 0.92)
pos_probs = np.clip(rng.beta(6.5, 1.5, size=n_pos), 0.12, 0.995)
s1_probs = np.concatenate([neg_probs, pos_probs])

s1_default = compute_stage1_metrics(s1_targets, s1_probs, threshold=0.50)
thr_res = find_sensitivity_threshold(s1_probs, s1_targets, target_sensitivity=0.98)
s1_calibrated = compute_stage1_metrics(s1_targets, s1_probs, threshold=thr_res.threshold)

print("=== Stage 1 Screening Comparison ===")
for label, m in [("Default (0.50)", s1_default), (f"Calibrated ({thr_res.threshold:.4f})", s1_calibrated)]:
    print(
        f"  {label:22s} | Sens={m.sensitivity:.4f} | Spec={m.specificity:.4f} | "
        f"NPV={m.npv:.4f} | BalAcc={m.balanced_accuracy:.4f} | "
        f"AUROC={m.auroc:.4f} | AUPRC={m.auprc:.4f}"
    )
print(f"  Target sensitivity achieved: {thr_res.achieved_target} (CM calibrated={s1_calibrated.confusion_matrix})")

p_s1_cm = plot_confusion_matrix(
    s1_calibrated.confusion_matrix,
    class_names=["no_tumor", "tumor"],
    output_path=scratch_dir / "stage1_confusion_matrix.png",
    title=f"Stage 1 Confusion Matrix (tau={thr_res.threshold:.3f})",
)
p_s1_roc = plot_roc_curves(
    s1_targets,
    s1_probs,
    class_names=["no_tumor", "tumor"],
    output_path=scratch_dir / "stage1_roc.png",
    title="Stage 1 Screening ROC Curve",
)
plot_paths["Stage 1 Confusion Matrix"] = p_s1_cm
plot_paths["Stage 1 ROC Curve"] = p_s1_roc

show_saved_plot(p_s1_cm, figsize=(5.5, 4.5))
show_saved_plot(p_s1_roc, figsize=(6.0, 4.8))

## 3. Stage 2 Subtype Calibration (`ECE`, `Brier Score`) & Abstention (`0.70`)

We fit `TemperatureScaler` on a validation split of overconfident 3-class logits (`glioma`, `meningioma`, `pituitary`), then evaluate the held-out test split before and after scaling using `compute_ece`, `compute_stage2_metrics(..., uncertainty_threshold=0.70)`, and `plot_calibration_curve`.

In [ ]:
stage2_classes = ["glioma", "meningioma", "pituitary"]

def make_imbalanced_stage2_logits(n_per_class: tuple[int, int, int], temp_distortion: float = 2.6) -> tuple[np.ndarray, np.ndarray]:
    labels = np.concatenate([
        np.full(n_per_class[0], 0, dtype=np.int64),
        np.full(n_per_class[1], 1, dtype=np.int64),
        np.full(n_per_class[2], 2, dtype=np.int64),
    ])
    base_logits = rng.normal(loc=0.0, scale=1.15, size=(len(labels), 3)).astype(np.float32)
    for idx, y in enumerate(labels):
        base_logits[idx, y] += 1.95
    return base_logits * temp_distortion, labels

val_logits, val_labels = make_imbalanced_stage2_logits((90, 45, 65))
test_logits, test_labels = make_imbalanced_stage2_logits((140, 70, 90))

probs_before = torch.softmax(torch.tensor(test_logits, dtype=torch.float32), dim=-1).numpy()
scaler = TemperatureScaler(init_temperature=1.0)
fitted_temp = scaler.fit(val_logits, val_labels, max_iter=50, lr=0.05)

abstain_before = apply_temperature_and_abstention(test_logits, temperature=1.0, uncertainty_threshold=0.70)
abstain_after = apply_temperature_and_abstention(test_logits, temperature=fitted_temp, uncertainty_threshold=0.70)

s2_before = compute_stage2_metrics(test_labels, probs_before, class_names=stage2_classes, uncertainty_threshold=0.70)
s2_after = compute_stage2_metrics(test_labels, abstain_after["probs"], class_names=stage2_classes, uncertainty_threshold=0.70)
ece_before = compute_ece(probs_before, test_labels, n_bins=10)
ece_after = compute_ece(abstain_after["probs"], test_labels, n_bins=10)

print(f"=== Stage 2 Temperature Scaling (fitted T = {fitted_temp:.4f}) ===")
print(f"  Macro-F1          : {s2_before.macro_f1:.4f} | Balanced Acc: {s2_before.balanced_accuracy:.4f}")
print(f"  ECE (M=10)        : {ece_before:.4f} -> {ece_after:.4f}")
print(f"  Brier Score       : {s2_before.brier_score:.4f} -> {s2_after.brier_score:.4f}")
print(f"  Abstention (<0.70): {s2_before.abstention_rate:.4f} -> {s2_after.abstention_rate:.4f}")
print(f"  Selective Acc     : {s2_before.selective_accuracy:.4f} -> {s2_after.selective_accuracy:.4f}")
print(f"  Selective MacroF1 : {s2_before.selective_macro_f1:.4f} -> {s2_after.selective_macro_f1:.4f}")

p_cal_before = plot_calibration_curve(
    test_labels,
    probs_before,
    output_path=scratch_dir / "stage2_reliability_before.png",
    title="Stage 2 Reliability Before Temp Scaling (T=1.0)",
)
p_cal_after = plot_calibration_curve(
    test_labels,
    abstain_after["probs"],
    output_path=scratch_dir / "stage2_reliability_after.png",
    title=f"Stage 2 Reliability After Temp Scaling (T={fitted_temp:.2f})",
)
p_s2_roc = plot_roc_curves(
    test_labels,
    abstain_after["probs"],
    class_names=stage2_classes,
    output_path=scratch_dir / "stage2_roc.png",
    title="Stage 2 OvR ROC Curves",
)
plot_paths["Stage 2 Reliability (Before)"] = p_cal_before
plot_paths["Stage 2 Reliability (After)"] = p_cal_after
plot_paths["Stage 2 ROC Curves"] = p_s2_roc

show_saved_plot(p_cal_before, figsize=(5.5, 4.8))
show_saved_plot(p_cal_after, figsize=(5.5, 4.8))

## 4. End-to-End `4x5` Clinical Routing Matrix (`compute_cascade_metrics`)

We combine Stage 1 screening (`tau = thr_res.threshold`) with calibrated Stage 2 subtyping + confidence abstention (`uncertainty_threshold = 0.70`) over a 4-class cohort (`no_tumor`, `glioma`, `meningioma`, `pituitary`), verifying `cascade_tumor_sensitivity` (which treats abstained true tumors as detected), `overall_4way_*` vs. `selective_4way_*`, and plotting the `4x5` routing matrix.

In [ ]:
cascade_classes = ["no_tumor", "glioma", "meningioma", "pituitary"]
# True labels: 120 healthy + 300 tumor slices from Stage 2 test set
n_healthy_cas = 120
true_4way = np.concatenate([
    np.zeros(n_healthy_cas, dtype=np.int64),
    test_labels + 1,
])

# Stage 1 probabilities for healthy + tumor slices
s1_probs_cas = np.concatenate([
    np.clip(rng.beta(1.8, 7.0, size=n_healthy_cas), 0.005, 0.92),
    np.clip(rng.beta(6.5, 1.5, size=len(test_labels)), 0.12, 0.995),
])

# False-positive healthy scans passed to Stage 2 get ambiguous Stage 2 logits
healthy_s2_logits = rng.normal(loc=0.0, scale=0.8, size=(n_healthy_cas, 3)).astype(np.float32)
all_s2_logits = np.concatenate([healthy_s2_logits, test_logits], axis=0)
s2_cas_res = apply_temperature_and_abstention(
    all_s2_logits,
    temperature=fitted_temp,
    uncertainty_threshold=0.70,
    class_names=stage2_classes,
)

routed_preds: list[str] = []
raw_4way_preds: list[str] = []
for idx in range(len(true_4way)):
    if s1_probs_cas[idx] < thr_res.threshold:
        routed_preds.append("no_tumor")
        raw_4way_preds.append("no_tumor")
    else:
        raw_cls = stage2_classes[int(s2_cas_res["pred_indices"][idx])]
        raw_4way_preds.append(raw_cls)
        routed_preds.append(str(s2_cas_res["labels"][idx]))

cascade_metrics = compute_cascade_metrics(
    true_labels=true_4way,
    pred_labels=routed_preds,
    raw_4way_preds=raw_4way_preds,
    class_names=cascade_classes,
)
cascade_no_raw = compute_cascade_metrics(
    true_labels=true_4way,
    pred_labels=routed_preds,
    class_names=cascade_classes,
)

print("=== End-to-End 4-Way Cascade Metrics ===")
print(f"  Cascade Tumor Sensitivity   : {cascade_metrics.cascade_tumor_sensitivity:.4f}")
print(f"  Cascade Healthy Specificity : {cascade_metrics.cascade_healthy_specificity:.4f}")
print(f"  Abstention Rate             : {cascade_metrics.abstention_rate:.4f}")
print(f"  Selective 4-Way Acc / F1    : {cascade_metrics.selective_4way_accuracy:.4f} / {cascade_metrics.selective_4way_macro_f1:.4f}")
print(f"  Overall 4-Way (raw_preds)   : {cascade_metrics.overall_4way_accuracy:.4f} / {cascade_metrics.overall_4way_macro_f1:.4f}")
print(f"  Overall 4-Way (abstain=err) : {cascade_no_raw.overall_4way_accuracy:.4f} / {cascade_no_raw.overall_4way_macro_f1:.4f}")

p_cas_routing = plot_confusion_matrix(
    cascade_metrics.routing_matrix_4x5,
    class_names=cascade_metrics.routing_columns,
    output_path=scratch_dir / "cascade_routing_4x5.png",
    title="End-to-End 4x5 Clinical Routing Matrix",
)
plot_paths["4x5 Clinical Routing Matrix"] = p_cas_routing
show_saved_plot(p_cas_routing, figsize=(7.0, 5.0))

## 5. Grad-CAM Attribution & Quantitative Mask Validation (`SimpleCNN` vs. `EfficientNet-B0` vs. `ResNet-50`)

We generate synthetic `(224, 224)` axial MRI slices with circular tumor lesions and exact binary ground-truth masks (distinguished by anatomical quadrant/row band per subtype). After briefly fitting `SimpleCNN`, `TimmClassifier("efficientnet_b0", pretrained=False)`, and `TimmClassifier("resnet50", pretrained=False)`, we:
1. Run `GradCAM` across all three CNN backbones (verifying clean gradient hooks without `inplace=True` conflicts across `Conv2d`, `conv_head`, and `layer4[-1]`).
2. Compute single-slice `compute_pointing_game_hit` and `compute_energy_inside_mask`.
3. Evaluate a masked `DataLoader` via `evaluate_gradcam_against_masks`.
4. Render `overlay_gradcam` and save `plot_gradcam_examples`.

In [ ]:
def make_synthetic_mri_with_masks(
    n_per_class: int,
    image_size: int = 224,
    seed: int = 100,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    local_rng = np.random.default_rng(seed)
    yy, xx = np.ogrid[:image_size, :image_size]
    cy_brain, cx_brain = image_size / 2.0, image_size / 2.0
    brain_ellipse = ((yy - cy_brain) / (0.38 * image_size)) ** 2 + ((xx - cx_brain) / (0.32 * image_size)) ** 2 <= 1.0

    # Distinct anatomical positions AND morphological patterns per subtype:
    # 0=glioma (superior-left, ring-enhancing rim with dark necrotic core)
    # 1=meningioma (mid-right convexity, large solid hyperintense disk)
    # 2=pituitary (inferior-central sella, compact bright nodule with subtle halo)
    centers = [(68.0, 84.0), (112.0, 150.0), (158.0, 112.0)]
    radii = [24.0, 22.0, 16.0]

    imgs, lbls, masks = [], [], []
    for cls_idx in range(3):
        cy_base, cx_base = centers[cls_idx]
        rad_base = radii[cls_idx]
        for _ in range(n_per_class):
            cy = cy_base + float(local_rng.uniform(-4.0, 4.0))
            cx = cx_base + float(local_rng.uniform(-4.0, 4.0))
            rad = rad_base + float(local_rng.uniform(-1.5, 1.5))

            slice_2d = np.zeros((image_size, image_size), dtype=np.float32)
            slice_2d[brain_ellipse] = 0.22
            slice_2d += local_rng.normal(0.0, 0.02, size=(image_size, image_size)).astype(np.float32)
            slice_2d[~brain_ellipse] *= 0.15

            dist2 = (yy - cy) ** 2 + (xx - cx) ** 2
            tumor_mask = dist2 <= rad ** 2
            if cls_idx == 0:
                # Ring-enhancing glioma: bright outer rim, darker necrotic center
                rim = tumor_mask & (dist2 >= (0.45 * rad) ** 2)
                core = dist2 < (0.45 * rad) ** 2
                slice_2d[rim] += 0.70
                slice_2d[core] += 0.15
            elif cls_idx == 1:
                # Homogeneous solid meningioma
                slice_2d[tumor_mask] += 0.62
            else:
                # Compact bright pituitary lesion with striped/cross texture
                slice_2d[tumor_mask] += 0.72
                stripe = tumor_mask & (((yy + xx) % 4) < 2)
                slice_2d[stripe] -= 0.25
            slice_2d = np.clip(slice_2d, 0.0, 1.0)

            rgb = np.stack([slice_2d, slice_2d, slice_2d], axis=0)
            imgs.append(rgb)
            lbls.append(cls_idx)
            masks.append(tumor_mask.astype(np.float32))

    return (
        torch.tensor(np.stack(imgs), dtype=torch.float32),
        torch.tensor(lbls, dtype=torch.long),
        torch.tensor(np.stack(masks), dtype=torch.float32),
    )

train_imgs, train_lbls, _ = make_synthetic_mri_with_masks(n_per_class=12, seed=101)
val_imgs, val_lbls, val_masks = make_synthetic_mri_with_masks(n_per_class=4, seed=202)

train_loader_gc = DataLoader(TensorDataset(train_imgs, train_lbls), batch_size=6, shuffle=True)
val_loader_gc = DataLoader(TensorDataset(val_imgs, val_lbls, val_masks), batch_size=6, shuffle=False)

def quick_fit(model: nn.Module, loader: DataLoader, epochs: int = 10, lr: float = 3e-3) -> None:
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

def eval_classifier(model: nn.Module, imgs: torch.Tensor, lbls: torch.Tensor) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(imgs), dim=-1).numpy()
    m = compute_stage2_metrics(lbls.numpy(), probs, class_names=stage2_classes)
    return {"accuracy": m.accuracy, "macro_f1": m.macro_f1, "balanced_accuracy": m.balanced_accuracy}

simple_cnn = SimpleCNN(num_classes=3, dropout=0.1)
effnet_cnn = TimmClassifier(
    backbone_name="efficientnet_b0",
    num_classes=3,
    pretrained=False,
    dropout=0.1,
)
resnet_cnn = TimmClassifier(
    backbone_name="resnet50",
    num_classes=3,
    pretrained=False,
    dropout=0.1,
)

quick_fit(simple_cnn, train_loader_gc, epochs=10, lr=3e-3)
quick_fit(effnet_cnn, train_loader_gc, epochs=8, lr=3e-3)
quick_fit(resnet_cnn, train_loader_gc, epochs=6, lr=1e-3)

cls_simple = eval_classifier(simple_cnn, val_imgs, val_lbls)
cls_effnet = eval_classifier(effnet_cnn, val_imgs, val_lbls)
cls_resnet = eval_classifier(resnet_cnn, val_imgs, val_lbls)

with GradCAM(simple_cnn) as cam_simple:
    gc_simple = evaluate_gradcam_against_masks(cam_simple, val_loader_gc, target_mode="predicted")

with GradCAM(effnet_cnn) as cam_effnet:
    gc_effnet = evaluate_gradcam_against_masks(cam_effnet, val_loader_gc, target_mode="predicted")
    # Single-sample Grad-CAM & overlay inspection for 1 sample per class (indices 0, 4, 8)
    example_indices = [0, 4, 8]
    example_imgs = [val_imgs[i] for i in example_indices]
    example_masks = [val_masks[i] for i in example_indices]
    example_cams = [cam_effnet.generate(val_imgs[i]) for i in example_indices]

with GradCAM(resnet_cnn) as cam_resnet:
    gc_resnet = evaluate_gradcam_against_masks(cam_resnet, val_loader_gc, target_mode="predicted")

print("=== Quantitative Grad-CAM Localization (Val Masked DataLoader) ===")
for name, cls_m, gc_m in [
    ("SimpleCNN", cls_simple, gc_simple),
    ("EfficientNet-B0", cls_effnet, gc_effnet),
    ("ResNet-50", cls_resnet, gc_resnet),
]:
    print(
        f"  {name:18s} | MacroF1={cls_m['macro_f1']:.3f} | "
        f"HitRate={gc_m['pointing_game_hit_rate']:.3f} | "
        f"MeanEnergyInMask={gc_m['mean_energy_in_mask']:.3f} +/- {gc_m['std_energy_in_mask']:.3f}"
    )

example_titles = []
for idx, cam_res, msk_t in zip(example_indices, example_cams, example_masks):
    msk_np = msk_t.numpy()
    hit = compute_pointing_game_hit(cam_res.heatmap, msk_np)
    energy = compute_energy_inside_mask(cam_res.heatmap, msk_np)
    true_name = stage2_classes[int(val_lbls[idx].item())]
    pred_name = stage2_classes[cam_res.predicted_class]
    example_titles.append(f"True: {true_name}\nPred: {pred_name} ({cam_res.confidence:.2f})\nHit={hit} | E={energy:.2f}")

sample_overlay = overlay_gradcam(val_imgs[0].permute(1, 2, 0).numpy(), example_cams[0].heatmap)
print(f"Rendered single overlay_gradcam shape: {sample_overlay.shape}, dtype={sample_overlay.dtype}")

p_gc_examples = plot_gradcam_examples(
    images=example_imgs,
    masks=example_masks,
    heatmaps=[c.heatmap for c in example_cams],
    titles=example_titles,
    output_path=scratch_dir / "gradcam_examples.png",
)
plot_paths["Grad-CAM Examples"] = p_gc_examples
show_saved_plot(p_gc_examples, figsize=(9.5, 8.5))

## 6. Model Ablation Comparison & Full Evaluation Report Generation

Finally, we plot an ablation chart (`plot_ablation_comparison`) comparing `SimpleCNN`, `EfficientNet-B0`, and `ResNet-50`, generate the complete Markdown + JSON report (`generate_evaluation_report`) inside our scratch directory, verify the output artifacts, and clean up the temporary directory.

In [ ]:
ablation_table = {
    "SimpleCNN": {
        "macro_f1": cls_simple["macro_f1"],
        "pointing_game_hit_rate": gc_simple["pointing_game_hit_rate"],
        "mean_energy_in_mask": gc_simple["mean_energy_in_mask"],
    },
    "EfficientNet-B0": {
        "macro_f1": cls_effnet["macro_f1"],
        "pointing_game_hit_rate": gc_effnet["pointing_game_hit_rate"],
        "mean_energy_in_mask": gc_effnet["mean_energy_in_mask"],
    },
    "ResNet-50": {
        "macro_f1": cls_resnet["macro_f1"],
        "pointing_game_hit_rate": gc_resnet["pointing_game_hit_rate"],
        "mean_energy_in_mask": gc_resnet["mean_energy_in_mask"],
    },
}

p_ablation = plot_ablation_comparison(
    results=ablation_table,
    metrics=["macro_f1", "pointing_game_hit_rate", "mean_energy_in_mask"],
    output_path=scratch_dir / "ablation_comparison.png",
    title="CNN Backbone Comparison: Classification & Localization",
)
plot_paths["Ablation Comparison"] = p_ablation
show_saved_plot(p_ablation, figsize=(7.0, 4.8))

report_path = generate_evaluation_report(
    output_dir=scratch_dir,
    stage1_metrics=s1_calibrated,
    stage2_metrics=s2_after,
    cascade_metrics=cascade_metrics,
    gradcam_metrics=gc_effnet,
    ablation_table=ablation_table,
    plot_paths=plot_paths,
    title="Milestone 3 Synthetic Walkthrough Evaluation Report",
)
json_path = scratch_dir / "metrics_summary.json"
assert report_path.exists() and report_path.stat().st_size > 0
assert json_path.exists() and json_path.stat().st_size > 0

summary_data = json.loads(json_path.read_text(encoding="utf-8"))
report_lines = report_path.read_text(encoding="utf-8").splitlines()
print(f"Generated report: {report_path.name} ({len(report_lines)} lines)")
print(f"Generated JSON  : {json_path.name} (sections={list(summary_data.keys())})")
print("\n--- Report Preview (first 28 lines) ---")
print("\n".join(report_lines[:28]))

scratch_ctx.cleanup()
print("\nTemporary scratch directory cleaned up cleanly.")